In [2]:
import torch

inputs = torch.tensor(
    [[0.72, 0.45, 0.31],
     [0.75, 0.20, 0.55],
     [0.30, 0.80, 0.40],
     [0.85, 0.35, 0.60],
     [0.55, 0.15, 0.75],
     [0.25, 0.20, 0.85]] #6x3
)

words = ['Dream', 'big', 'and', 'work', 'for', 'it']

We want to generate the context vector for the 2nd token

In [8]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2 #C
print(d_in)
print(x_2)
#3x2

3
tensor([0.7500, 0.2000, 0.5500])


Randomly initializing Wq, Wk, Wv matrices

In [5]:
torch.manual_seed(123) #Reproducibility
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [6]:
print(W_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [7]:
print(W_key)

Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])


In [9]:
print(W_value)

Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


If you want to see how the second vector from the input embedding getting transformed, here's how

In [10]:
query_2 = x_2 @ W_query #1x3 * 3x2 = 1x2
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([0.3131, 1.0017])
tensor([0.3126, 0.6001])
tensor([0.1852, 0.6829])


In [ ]:
# query =
# [0.7500, 0.2000, 0.5500]
# x [[0.2961, 0.5166],
#         [0.2517, 0.6886],
#         [0.0740, 0.8665]])

Calculating Q, K and V using X, Wq, Wk, Wv

In [11]:
keys = inputs @ W_key #6x3 x 3x2
queries = inputs @ W_query
values = inputs @ W_value
print(keys)
print(queries)
print(values)


tensor([[0.2789, 0.6137],
        [0.3126, 0.6001],
        [0.3143, 0.8867],
        [0.3697, 0.7536],
        [0.3392, 0.6807],
        [0.3389, 0.7549]])
tensor([[0.3494, 0.9504],
        [0.3131, 1.0017],
        [0.3198, 1.0524],
        [0.3842, 1.2000],
        [0.2561, 1.0373],
        [0.1872, 1.0034]])
tensor([[0.2336, 0.5789],
        [0.1852, 0.6829],
        [0.3232, 0.7113],
        [0.2462, 0.8042],
        [0.1780, 0.7890],
        [0.1830, 0.8328]])


Keys corresponding8 to second token and the attention of second token to itself

In [12]:
keys_2 = keys[1] #A
attn_score_22 = query_2.dot(keys_2) #diagonal of 2nd row 2nd column of attention matrix
print(attn_score_22)


tensor(0.6990)


All attention scores for query number 2

In [13]:
attn_scores_2 = query_2 @ keys.T #6x6
print(attn_scores_2)


tensor([0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624])


Attention scores (not WEIGHS) matrix

In [14]:
attn_scores = queries @ keys.T # omega
print(attn_scores)

tensor([[0.6807, 0.6795, 0.9526, 0.8454, 0.7654, 0.8359],
        [0.7021, 0.6990, 0.9867, 0.8707, 0.7880, 0.8624],
        [0.7350, 0.7315, 1.0337, 0.9113, 0.8248, 0.9029],
        [0.8436, 0.8402, 1.1848, 1.0464, 0.9471, 1.0361],
        [0.7080, 0.7025, 1.0003, 0.8764, 0.7929, 0.8699],
        [0.6680, 0.6606, 0.9486, 0.8254, 0.7465, 0.8210]])


Scale by 1/sqrt(d) and then take softmaxx

In [16]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / (d_k ** (1/2)), dim=-1)
print(attn_weights_2)
print(d_k)

tensor([0.1531, 0.1528, 0.1873, 0.1725, 0.1627, 0.1715])
2


In [19]:
d_k = keys.shape[-1]
attn_scores_scaled = queries @ keys.T / (d_k ** (1/2))
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.1536, 0.1534, 0.1861, 0.1725, 0.1630, 0.1714],
        [0.1531, 0.1528, 0.1873, 0.1725, 0.1627, 0.1715],
        [0.1525, 0.1521, 0.1884, 0.1728, 0.1625, 0.1717],
        [0.1505, 0.1501, 0.1915, 0.1737, 0.1619, 0.1724],
        [0.1530, 0.1524, 0.1881, 0.1724, 0.1625, 0.1716],
        [0.1538, 0.1530, 0.1875, 0.1719, 0.1625, 0.1713]])


Softmax peaks when the numbers are scaled

In [ ]:
import torch

#Define the

Context vector corresponding to 2nd input token

In [20]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.2274, 0.7362])


In [22]:
context_vec = attn_weights @ values
print(context_vec)

tensor([[0.2273, 0.7361],
        [0.2274, 0.7362],
        [0.2276, 0.7363],
        [0.2280, 0.7368],
        [0.2275, 0.7362],
        [0.2275, 0.7360]])


Python class for doing this whole operation